# Act II — Building better graphs

A schema gives the graph its **shape** (Act I). But the *details* of what goes into that shape
still vary: `Eggs` vs `egg`, cups vs grams, `Minced Garlic` vs `garlic clove`. Duplicates like
that silently fragment your answers — "find every recipe with garlic" only finds the ones that
happened to spell it the same way.

Three techniques from the talk, in order:

1. **Ontology in the prompt** — tell the extractor *what* to put in the shape: canonical names,
   standard (metric) units. Recipe model **v1 → v2**.
2. **Simple entity matching** — a synonym / index table that maps surface forms onto one
   canonical node. Cheap and precise when you already know your domain's vocabulary.
3. **Better entity matching** — a hybrid of an embedding model (meaning) and a lexical score
   (spelling), for the terms you *didn't* know in advance.

Everything runs **offline**: extraction replays from the committed cache in `data/cache/`.
Section 3 needs the optional `vector` extra (see the note there).


## 1. Ontology in the prompt — v1 → v2

**Principle:** *a schema provides shape; the ontology describes what should go into it.*

Same `Recipe` schema, but the system prompt now carries the rules: convert units to metric
(`1 cup → 240 ml`), lowercase singular ingredient names, strip brands and loose adjectives.
Those instructions matter to the model as much as the schema does.

Both extractors replay from cache (separate `recipe-v1` / `recipe-v2` cache tags) — no LLM call.


In [ ]:
from graphtools.data import load_hero_texts
from graphtools.extract import extract_recipe, extract_recipe_v2

hero_texts = load_hero_texts()  # 14 real recipes (TheMealDB), our deterministic corpus

# Beef Lo Mein is rich in cups/tbsp/lb units and Title-Case plurals, so v1 -> v2 is vivid.
hero_title, hero_text = next((t, x) for t, x in hero_texts if t == "Beef Lo Mein")
v1 = extract_recipe(hero_text)      # schema only
v2 = extract_recipe_v2(hero_text)   # schema + ontology rules in the prompt

print(f"=== {hero_title}:  v1 (schema only)  ->  v2 (schema + ontology) ===\n")
print(f"  {'ingredient (v1)':22} {'qty/unit (v1)':14} | {'ingredient (v2)':22} {'qty/unit (v2)':14}")
print(f"  {'-'*22} {'-'*14} | {'-'*22} {'-'*14}")
for a, b in zip(v1.ingredients, v2.ingredients):
    u1 = f"{a.quantity if a.quantity is not None else '-'} {a.unit or '-'}"
    u2 = f"{b.quantity if b.quantity is not None else '-'} {b.unit or '-'}"
    flag = "  <- fixed" if (a.unit != b.unit or a.name.lower() != b.name.lower()) else ""
    print(f"  {a.name:22} {u1:14} | {b.name:22} {u2:14}{flag}")


In [ ]:
# The headline: cooking units became metric; Title-Case plurals became clean names.
units_v1 = sorted({i.unit for i in v1.ingredients if i.unit})
units_v2 = sorted({i.unit for i in v2.ingredients if i.unit})
print(f"units  v1: {units_v1}")
print(f"units  v2: {units_v2}\n")

print("a few name + unit fixes the ontology-in-the-prompt bought us:")
for a, b in zip(v1.ingredients, v2.ingredients):
    if a.unit != b.unit or a.name.lower() != b.name.lower():
        print(f"  {a.name} ({a.quantity} {a.unit})  ->  {b.name} ({b.quantity} {b.unit})")


The best prompt in the world isn't bulletproof, though: `fresh eggs`, `large eggs`, `organic eggs`
will still leak through. So we also match entities *as we build the graph*.

## 2. Simple entity matching — a synonym table

**Principle:** *get your nodes right and the relationships fall into place.*

`normalise_ingredient` maps surface forms onto canonical names using a curated synonym table
(`data/ontology/ingredients.yaml`) with a fuzzy-match fallback. Across the whole hero set it
collapses plurals, prep adjectives and synonyms onto shared nodes — so the `CONTAINS` edges from
different recipes finally point at the *same* ingredient.


In [ ]:
from collections import defaultdict

from graphtools.resolve import normalise_ingredient

# Every ingredient surface form the v1 extractor produced across the hero set.
recipes_v1 = [extract_recipe(t) for _, t in hero_texts]
surface_forms = [ing.name for r in recipes_v1 for ing in r.ingredients]

distinct_before = set(surface_forms)
distinct_after = {normalise_ingredient(s) for s in surface_forms}
print(f"{len(surface_forms)} ingredient mentions across {len(recipes_v1)} recipes")
print(f"distinct surface forms (before): {len(distinct_before)}")
print(f"distinct canonical    (after) : {len(distinct_after)}")
print(f"-> {len(distinct_before) - len(distinct_after)} duplicate nodes collapsed\n")

# Show the merges concretely: which surface forms folded onto each canonical node.
groups = defaultdict(set)
for s in distinct_before:
    groups[normalise_ingredient(s)].add(s)
merged = {k: v for k, v in groups.items() if len(v) > 1}
print(f"{len(merged)} canonical ingredients absorbed >1 surface form:")
for canon, variants in sorted(merged.items()):
    print(f"  {canon:14} <- {sorted(variants)}")


**See it, don't just count it.** A ten-node drop is invisible in a 250-node hairball, so here is
the same move at a legible scale — the five recipes from the talk's *simple entity matching* slide,
which each spell the shared pantry items differently (`Minced Garlic` / `garlic` / `Garlic Clove`;
`Cumin` / `Cumin seeds`; `Oil` / `vegetable oil`), projected to just the ingredients that drift.

Before matching, every recipe brings its own spelling and the recipes barely touch. After, the
duplicates collapse onto one shared hub each and the recipes become a woven web. *That's the
relationships falling out of matched entities.* (The animated before/after is in the deck.)


In [ ]:
from graphtools.focus import RUNG3_RECIPES, drift_focus_graph

before = drift_focus_graph(RUNG3_RECIPES, normalise=False)
after = drift_focus_graph(RUNG3_RECIPES, normalise=True)


def ingredient_nodes(g):
    return [n for n, d in g.nodes(data=True) if d["kind"] == "ingredient"]


print(f"{len(RUNG3_RECIPES)} recipes, drift-focus projection:")
print(f"  ingredient nodes  before -> after :  {len(ingredient_nodes(before))} -> {len(ingredient_nodes(after))}")

# The payoff: recipes now SHARE ingredient nodes. An ingredient with in-degree >= 2 is a hub
# that several recipes point at — the relationships that appear once entities are matched.
hubs = sorted(after.nodes[n]["label"] for n in ingredient_nodes(after) if after.in_degree(n) >= 2)
print(f"  shared ingredient hubs (>=2 recipes): {hubs}")


## 3. Better entity matching — hybrid (meaning + spelling)

**Principle:** *hybrid approaches often give the best result.* A synonym table handles what you
know; an embedding model handles the long tail you don't, without listing every variant up front.

`hybrid_lookup` scores each candidate as `0.6 × cosine(embedding) + 0.4 × token-sort ratio` and
returns the best first. Here the candidates are the canonical names we already have (synonym table
+ graph), and the queries are spellings that appear in **none** of the synonym lists.

> **Needs the `vector` extra:** `uv sync --extra dev --extra vector`. The first run downloads the
> `all-MiniLM-L6-v2` sentence-transformer model (~90 MB) — the only network call in the pack, and
> only if you run this cell.


In [ ]:
import yaml
from graphtools.resolve import hybrid_lookup

# Candidates: the canonical ingredient names we already have (synonym table + the graph's nodes).
ontology = yaml.safe_load(open("../data/ontology/ingredients.yaml"))
candidates = sorted(set(ontology) | set(distinct_after))
print(f"{len(candidates)} canonical candidates\n")

# Spellings that appear in no synonym list — the match has to be earned by meaning + spelling.
for query in ["AP flour", "cilantro", "confectioners sugar", "minced beef", "garbanzo"]:
    print(f"  {query:20} -> {hybrid_lookup(query, candidates, top=3)}")


Four out of five: `AP flour → flour`, `cilantro → coriander`, `confectioners sugar → icing sugar`,
`minced beef → lean minced beef`. And one honest miss — a small general-purpose embedding model
doesn't know that *garbanzo* is a chickpea, so the top hit is noise. That's the real lesson of
**hybrid**: cheap exact matches from the table first, embeddings for the long tail, and a bigger or
domain-tuned embedding model when the tail matters. (The deck's scatter plot used a larger model,
which does place garbanzo with chickpea.)

---

We now have a well-structured graph, curated content in it, and entities matched *before* new
nodes are created. **Act III** looks at what we can do with it.

**Now you try:** add a synonym to `data/ontology/ingredients.yaml` and re-run section 2 — does the
collapse count change? Then find another query `hybrid_lookup` gets wrong; would a threshold on the
score (rather than "best first") have caught it?
